In [88]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pandas as pd
import re
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

In [89]:
protein = "VCAM1"
synapse_type = "VGLUT1-PSD95"

In [92]:
results_file = f"/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/{protein}/{protein}-LacZ_{synapse_type}_output_data/metric_results.csv"
results = pd.read_csv(results_file)
results.head(10)

,Unnamed: 0,img_filename,local_peak_colocalized_spots,local_peak_colocalized_spots_rot,overlap_coeff,overlap_coeff_rot,overlap_um2,overlap_um2_rot,pearson_cor,pvalue,...,postsynapse_image_mfi,pre_puncta_density_per_100_um2,post_puncta_density_per_100_um2,pre_staining_area_um2,post_staining_area_um2,pre_mean_puncta_size_um2,post_mean_puncta_size_um2,gRNA,hippocampal_layer,Brain
0,0,CRISPR-Exp5_IHC-Exp2_Brain-7_section-1_VCAM1-g...,14,3,0.2807456,0.0843337,36.0851774,10.8396949,0.3028476,0.0,...,232.4776859,18.7368305,17.1937974,155.8091135,94.2160656,0.4582621,0.3019746,VCAM1-gRNA,DG Hilus,Brain-7
1,1,CRISPR-Exp5_IHC-Exp2_Brain-5_section-2_LacZ-gR...,277,101,0.1818718,0.0813838,24.2657867,10.8584369,0.2733444,0.0,...,240.0566278,49.5423842,40.7250522,162.0110137,103.1185185,0.1802125,0.1395379,LacZ-gRNA,CA3 SO,Brain-5
2,2,CRISPR-Exp4_IHC-Exp2_Brain-4_section-2_VCAM1-g...,243,109,0.1209427,0.0640370,14.2456286,7.5428056,0.1797840,0.0,...,300.4140611,55.4389750,77.3169801,86.1229265,132.5877677,0.0856093,0.0945030,VCAM1-gRNA,CA1 SO,Brain-4
3,3,CRISPR-Exp4_IHC-Exp2_Brain-4-2_section-3_VCAM1...,21,3,0.4007573,0.1246374,81.1136994,25.2267405,0.4754852,0.0,...,227.0595893,22.2637633,26.3968877,195.0582823,202.6147184,0.4828175,0.4229952,VCAM1-gRNA,CA3 SL,Brain-4-2
4,4,CRISPR-Exp5_IHC-Exp2_Brain-5_section-3_LacZ-gR...,42,3,0.3267824,0.0864053,49.5334188,13.0972548,0.3671362,0.0,...,202.9020050,24.8538546,16.0916309,222.4625031,95.7835788,0.4932650,0.3280260,LacZ-gRNA,DG Hilus,Brain-5
5,5,CRISPR-Exp5_IHC-Exp2_Brain-5_section-1_VCAM1-g...,111,35,0.1264367,0.0437623,11.5927828,4.0124932,0.2377350,0.0,...,232.6982472,34.0018366,32.6792368,98.4739086,78.4063314,0.1596011,0.1322198,VCAM1-gRNA,DG ML,Brain-5
6,6,CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_LacZ-gR...,164,54,0.1635020,0.0531378,17.1012288,5.5578568,0.2662463,0.0,...,192.8833162,36.4266029,38.5758275,104.8461908,96.3799154,0.1586175,0.1376856,LacZ-gRNA,CA3 SR,Brain-4
7,7,CRISPR-Exp5_IHC-Exp2_Brain-5_section-3_LacZ-gR...,197,84,0.1477206,0.0547041,15.4314864,5.7146081,0.2614413,0.0,...,225.3246471,42.0476520,36.2061696,121.1210675,83.7478031,0.1587432,0.1274700,LacZ-gRNA,CA3 SO,Brain-5
8,8,CRISPR-Exp4_IHC-Exp2_Brain-4_section-1_LacZ-gR...,379,219,0.1087563,0.0568132,10.6914627,5.5851179,0.1859585,0.0,...,276.3837538,48.6055427,61.8315407,77.7588802,100.3395902,0.0881620,0.0894292,LacZ-gRNA,CA1 SO,Brain-4
9,9,CRISPR-Exp5_IHC-Exp2_Brain-7_section-2_VCAM1-g...,35,13,0.3318316,0.1189911,61.8639551,22.1837202,0.3163351,0.0,...,290.3600313,22.9801716,23.2006049,213.6946518,152.2140559,0.5124572,0.3615536,VCAM1-gRNA,DG Hilus,Brain-7


In [94]:
# Add a new column 'section' by extracting the "section-<number>" part from 'img_filename'
results['section'] = results['img_filename'].apply(lambda x: re.search(r'section-\d+', x).group(0) if re.search(r'section-\d+', x) else None)

# Select the relevant columns
results_mfi = results[['presynapse_image_mfi', 'gRNA', 'hippocampal_layer', 'section', 'Brain']]

results_mfi.head(10)

,presynapse_image_mfi,gRNA,hippocampal_layer,section,Brain
0,442.3194247,VCAM1-gRNA,DG Hilus,section-1,Brain-7
1,1589.7434762,LacZ-gRNA,CA3 SO,section-2,Brain-5
2,742.1220367,VCAM1-gRNA,CA1 SO,section-2,Brain-4
3,493.6974500,VCAM1-gRNA,CA3 SL,section-3,Brain-4-2
4,857.1568904,LacZ-gRNA,DG Hilus,section-3,Brain-5
5,899.5746396,VCAM1-gRNA,DG ML,section-1,Brain-5
6,835.5293111,LacZ-gRNA,CA3 SR,section-1,Brain-4
7,1150.6024550,LacZ-gRNA,CA3 SO,section-3,Brain-5
8,1061.3039368,LacZ-gRNA,CA1 SO,section-1,Brain-4
9,594.6287821,VCAM1-gRNA,DG Hilus,section-2,Brain-7


In [95]:
scaler = StandardScaler()
results_mfi['presynapse_image_mfi_scaled'] = scaler.fit_transform(results_mfi[['presynapse_image_mfi']])

/var/folders/p5/hzbdgkws2lvg79nyhlq_g_sm0000gn/T/ipykernel_86018/3161662538.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_mfi['presynapse_image_mfi_scaled'] = scaler.fit_transform(results_mfi[['presynapse_image_mfi']])


In [96]:
results_mfi.head(10)

,presynapse_image_mfi,gRNA,hippocampal_layer,section,Brain,presynapse_image_mfi_scaled
0,442.3194247,VCAM1-gRNA,DG Hilus,section-1,Brain-7,-1.1613352
1,1589.7434762,LacZ-gRNA,CA3 SO,section-2,Brain-5,2.6986663
2,742.1220367,VCAM1-gRNA,CA1 SO,section-2,Brain-4,-0.1527817
3,493.6974500,VCAM1-gRNA,CA3 SL,section-3,Brain-4-2,-0.9884965
4,857.1568904,LacZ-gRNA,DG Hilus,section-3,Brain-5,0.2342023
5,899.5746396,VCAM1-gRNA,DG ML,section-1,Brain-5,0.3768980
6,835.5293111,LacZ-gRNA,CA3 SR,section-1,Brain-4,0.1614458
7,1150.6024550,LacZ-gRNA,CA3 SO,section-3,Brain-5,1.2213703
8,1061.3039368,LacZ-gRNA,CA1 SO,section-1,Brain-4,0.9209648
9,594.6287821,VCAM1-gRNA,DG Hilus,section-2,Brain-7,-0.6489576


In [80]:
# Create a combined key for the nested random effects as needed
results_mfi['Brain_gRNA'] = results_mfi['Brain'] + ':' + results_mfi['gRNA']
results_mfi['Brain_section_layer'] = results_mfi['Brain'] + ':' + results_mfi['section'] + ':' + results_mfi['hippocampal_layer']

/var/folders/p5/hzbdgkws2lvg79nyhlq_g_sm0000gn/T/ipykernel_86018/1634473522.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_mfi['Brain_gRNA'] = results_mfi['Brain'] + ':' + results_mfi['gRNA']
/var/folders/p5/hzbdgkws2lvg79nyhlq_g_sm0000gn/T/ipykernel_86018/1634473522.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  results_mfi['Brain_section_layer'] = results_mfi['Brain'] + ':' + results_mfi['section'] + ':' + results_mfi['hippocampal_layer']


In [97]:
results_mfi.head(10)

,presynapse_image_mfi,gRNA,hippocampal_layer,section,Brain,presynapse_image_mfi_scaled
0,442.3194247,VCAM1-gRNA,DG Hilus,section-1,Brain-7,-1.1613352
1,1589.7434762,LacZ-gRNA,CA3 SO,section-2,Brain-5,2.6986663
2,742.1220367,VCAM1-gRNA,CA1 SO,section-2,Brain-4,-0.1527817
3,493.6974500,VCAM1-gRNA,CA3 SL,section-3,Brain-4-2,-0.9884965
4,857.1568904,LacZ-gRNA,DG Hilus,section-3,Brain-5,0.2342023
5,899.5746396,VCAM1-gRNA,DG ML,section-1,Brain-5,0.3768980
6,835.5293111,LacZ-gRNA,CA3 SR,section-1,Brain-4,0.1614458
7,1150.6024550,LacZ-gRNA,CA3 SO,section-3,Brain-5,1.2213703
8,1061.3039368,LacZ-gRNA,CA1 SO,section-1,Brain-4,0.9209648
9,594.6287821,VCAM1-gRNA,DG Hilus,section-2,Brain-7,-0.6489576


In [99]:
# for decimals
pd.set_option('display.precision', 7)

# Define the mixed-effects model
model = smf.mixedlm(
    "presynapse_image_mfi_scaled ~ gRNA * hippocampal_layer",   # Fixed effects formula
    data=results_mfi,
    groups="Brain",                                            # Random intercept for Brain
    re_formula="~gRNA"                                           # Include random intercept
)

# Fit the model using REML
model_results = model.fit(reml=True)

# Print the summary of the model
print(model_results.summary())

# To get the variance components
print("\nVariance Components:")
print(model_results.cov_re)

                           Mixed Linear Model Regression Results
Model:                  MixedLM       Dependent Variable:       presynapse_image_mfi_scaled
No. Observations:       176           Method:                   REML                       
No. Groups:             4             Scale:                    0.2562                     
Min. group size:        32            Log-Likelihood:           -141.9095                  
Max. group size:        48            Converged:                Yes                        
Mean group size:        44.0                                                               
-------------------------------------------------------------------------------------------
                                                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------------------------
Intercept                                        -1.154    0.240 -4.814 0.000 -1.624 -0.684
gRNA[T.VCAM1-gR

/Users/cgeyskens/miniconda3/envs/synapse-counting-2/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/cgeyskens/miniconda3/envs/synapse-counting-2/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
/Users/cgeyskens/miniconda3/envs/synapse-counting-2/lib/python3.11/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [84]:
# Get the summary of the results
summary = model_results.summary()

# Extract coefficients and p-values into a DataFrame
coefficients = model_results.params
p_values = model_results.pvalues

adj_pvals = multipletests(p_values, method='fdr_bh')[1]
significance = ['Significant' if p < 0.05 else 'Not Significant' for p in adj_pvals]


# Create a DataFrame for coefficients and p-values
coefficients_table = pd.DataFrame({
    'Variable': coefficients.index,
    'Coefficient': coefficients,
    'P > z': p_values,
    "adj_pvals": adj_pvals,
    "significance": significance
})

# Reset index for better formatting (optional)
coefficients_table.reset_index(drop=True, inplace=True)

# Display the coefficients table
coefficients_table.head(20)

,Variable,Coefficient,P > z,adj_pvals,significance
0,Intercept,-1.1525867,7.9757980e-07,1.8942520e-06,Significant
1,gRNA[T.VCAM1-gRNA],0.0105405,9.5356977e-01,9.5356977e-01,Not Significant
2,hippocampal_layer[T.CA1 SO],1.7886114,1.2114634e-16,1.1508902e-15,Significant
3,hippocampal_layer[T.CA1 SR],1.0973454,3.7517086e-07,1.0183209e-06,Significant
4,hippocampal_layer[T.CA3 SL],1.5665134,4.0590939e-13,2.5707595e-12,Significant
5,hippocampal_layer[T.CA3 SO],3.3714960,6.0925086e-55,1.1575766e-53,Significant
6,hippocampal_layer[T.CA3 SR],1.3904966,1.2062118e-10,5.7295062e-10,Significant
7,hippocampal_layer[T.DG Hilus],0.9678248,7.4151026e-06,1.5654105e-05,Significant
8,hippocampal_layer[T.DG ML],1.1398236,1.3071984e-07,4.9673539e-07,Significant
9,gRNA[T.VCAM1-gRNA]:hippocampal_layer[T.CA1 SO],-0.2836247,2.5363945e-01,3.1264555e-01,Not Significant
